# CS13 TRN100k Reference Mesh Reconstruction

Reconstruct and quality-control the reference CS13 surface mesh from a deterministic 100,000-cell TRN point cloud.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


# CS13 TRN100k mesh reconstruction — reference profile

This notebook reconstructs **only the mesh** from the deterministic TRN 100,000-cell CS13 point cloud. It intentionally stops before celltype or marker-gene plotting so the mesh can be reviewed first. Existing accepted outputs are never overwritten.


## Reference parameters

The geometry-driving parameters follow the supplied reference call: `alpha=0.6`, `cs_method='marching_cube'`, `mc_scale_factor=0.98`, `smooth=5500`, and `scale_factor=1.0`. `alpha` is stored as mesh color opacity metadata; the preview below uses the established 0.20 overlay opacity. For the 100k point cloud, `dist_sample_num=2000` and a fixed seed prevent Spateo from allocating a 100,000 × 100,000 distance matrix.


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import anndata as ad
import pyvista as pv
from scipy.spatial import cKDTree

os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")
pv.OFF_SCREEN = True
import spateo as st

print("spateo", st.__version__)
print("pyvista", pv.__version__)


In [ ]:
RUN_RECONSTRUCTION = False  # Set True only after choosing a NEW empty output directory.
RNG_SEED = 20260814
DIST_SAMPLE_NUM = 2000
TRN_H5AD = Path(
    "/DATA/User/gaomohan/figures/figure2/b/cs13_trn100k_3d_v1/sampling/CS13.trn100k.h5ad"
)
OUTPUT_DIR = Path(
    "/DATA/User/gaomohan/figures/figure2/b/cs13_trn100k_3d_v1/models_reference_construct_surface_v1"
)
MESH_PATH = OUTPUT_DIR / "cs13_trn100k_mesh_reference_profile.vtk"
REFERENCE_CPO = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]
BACKGROUND = np.array([247, 248, 248], dtype=float) / 255.0


## Load TRN100k and construct the annotated point cloud

The point cloud keeps the TRN subset's `obs_index` and `celltype` labels. No additional sampling or point deletion occurs here.


## Load and validate data


## Construct the point-cloud model


In [ ]:
adata = ad.read_h5ad(TRN_H5AD)
assert adata.n_obs == 100000, adata.n_obs
assert "celltype" in adata.obs
assert "spatial" in adata.obsm

cs13_sub_pc, celltype_cmap = st.tdr.construct_pc(
    adata=adata,
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="rainbow",
)
print("TRN points:", cs13_sub_pc.n_points)


## Reconstruct mesh with the supplied profile

The call below is the reference call with only the deterministic `dist_sample_num` guard added. Change `OUTPUT_DIR` to a new version and set `RUN_RECONSTRUCTION=True` before rerunning.


## Reconstruct the surface mesh


In [ ]:
if RUN_RECONSTRUCTION:
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        raise FileExistsError(f"Refusing to overwrite non-empty output: {OUTPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.random.seed(RNG_SEED)
    cs13_mesh, cs13_inside_pc, _ = st.tdr.construct_surface(
        pc=cs13_sub_pc,
        key_added="tissue",
        alpha=0.6,
        cs_method="marching_cube",
        cs_args={"mc_scale_factor": 0.98, "dist_sample_num": DIST_SAMPLE_NUM},
        smooth=5500,
        scale_factor=1.00,
    )
    cs13_mesh = cs13_mesh.extract_surface().triangulate().clean()
    st.tdr.save_model(cs13_mesh, str(MESH_PATH))
else:
    cs13_mesh = pv.read(MESH_PATH).extract_surface().triangulate().clean()

print(cs13_mesh)


## Geometry QC

The review gate reports connectivity, open edges, and point-cloud fidelity. `bridge_vertex_fraction` is the fraction of mesh vertices farther than 240 µm from the TRN point cloud and is used only as a warning metric, not an automatic acceptance rule.


In [ ]:
mesh_points = np.asarray(cs13_mesh.points, dtype=float)
cloud_points = np.asarray(cs13_sub_pc.points, dtype=float)
vertex_to_cloud = cKDTree(cloud_points).query(mesh_points, k=1, workers=-1)[0]
cloud_to_vertex = cKDTree(mesh_points).query(cloud_points, k=1, workers=-1)[0]
bodies = list(cs13_mesh.split_bodies())
boundary = cs13_mesh.extract_feature_edges(
    boundary_edges=True, feature_edges=False, manifold_edges=False, non_manifold_edges=False
)
qc = {
    "n_points": int(cs13_mesh.n_points),
    "n_cells": int(cs13_mesh.n_cells),
    "n_bodies": len(bodies),
    "n_open_edges": int(cs13_mesh.n_open_edges),
    "n_boundary_loops": len(list(boundary.split_bodies())),
    "is_manifold": bool(cs13_mesh.is_manifold),
    "vertex_to_cloud_p95": float(np.quantile(vertex_to_cloud, 0.95)),
    "cloud_to_vertex_p95": float(np.quantile(cloud_to_vertex, 0.95)),
    "bridge_vertex_fraction_240um": float(np.mean(vertex_to_cloud > 240.0)),
}
print(json.dumps(qc, indent=2))


## Fixed-camera review renders

The first render shows the surface alone. The second uses a 0.20-opacity smooth shell over all 100,000 TRN points, matching the established presentation style.


In [ ]:
def render(mesh, pointcloud=None, output=None):
    plotter = pv.Plotter(off_screen=True, window_size=(1200, 1200))
    plotter.set_background(BACKGROUND)
    plotter.add_mesh(
        mesh,
        color="gainsboro",
        opacity=0.20 if pointcloud is not None else 1.0,
        smooth_shading=True,
        ambient=0.4,
        diffuse=0.65,
        specular=0.1,
        show_scalar_bar=False,
    )
    if pointcloud is not None:
        rgba_key = next((k for k in pointcloud.point_data.keys() if k.endswith("_rgba")), None)
        if rgba_key:
            plotter.add_mesh(
                pointcloud,
                scalars=rgba_key,
                rgba=True,
                style="points",
                model_size=1.2,
                opacity=0.42,
                render_points_as_spheres=False,
                show_scalar_bar=False,
            )
    plotter.camera_position = REFERENCE_CPO
    plotter.reset_camera_clipping_range()
    plotter.screenshot(str(output), transparent_background=False)
    plotter.close()

render(cs13_mesh, output=OUTPUT_DIR / "cs13_trn100k_mesh_reference_profile.png")
render(
    cs13_mesh,
    pointcloud=cs13_sub_pc,
    output=OUTPUT_DIR / "cs13_trn100k_mesh_reference_profile_pointcloud_overlay.png",
)


## Review gate

Stop here and inspect the VTK mesh plus the two PNG renders. Do not run celltype or marker-gene plotting until this mesh is explicitly accepted.
